# EPITECH Workshop: Training AI Models with data.gouv & scikit-learn

## A Practical Introduction to Machine Learning

**Duration**: 3-4 hours  
**Level**: Beginner to Intermediate  
**Goal**: Build and train your first AI model using real open data

---

### What You'll Learn Today:
- Why data.gouv is a treasure for AI training  
- How to find and load real-world datasets  
- How to prepare data for machine learning  
- How to train a basic AI model with scikit-learn  
- How to evaluate and interpret your model  

**Let's get started!**

## Section 1: Understanding data.gouv as a Data Source for AI Training

### What is data.gouv?

**data.gouv.fr** is the French government's open data portal. It's a platform where:
- French public administrations publish their datasets
- Data is freely available under open licenses
- Thousands of datasets cover diverse domains

### Why data.gouv is Perfect for Training AI Models

#### 1. **Real-World Data**
- Data comes from actual government operations
- Used by real institutions (hospitals, municipalities, agencies)
- Reflects real-world patterns and complexities

#### 2. **High Quality & Reliability**
- Data is cleaned and maintained by official organizations
- Good documentation and metadata
- Consistent formats (CSV, JSON, etc.)

#### 3. **Diversity of Domains**
- **Housing**: Property prices, real estate transactions
- **Health**: COVID-19 data, disease statistics
- **Transportation**: Traffic, public transport usage
- **Economy**: Employment, business statistics
- **Environment**: Air quality, water quality
- **Education**: School statistics, graduation rates
- **Public Services**: Police data, utilities usage

#### 4. **Completely Free & Legal**
- No licensing fees
- Open licenses (typically CC-BY or ODBL)
- Can use for commercial projects
- Perfect for learning!

### Examples of Datasets Available
- Housing prices in major French cities
- Local economic data by region
- Environmental pollution levels
- Public transport statistics
- Social indicators
- Tourism data
- Crime statistics

### Learning Value
Using real data teaches you:
- How data looks in production
- Real data quality challenges
- How to handle messy data
- True performance expectations
- Real-world problem-solving skills

**Today, we'll use a housing-related dataset from data.gouv to build our first classifier!**

## Section 2: Fetching and Loading Data from data.gouv

### Today's Dataset: Housing Data

For this workshop, we'll use a dataset about housing properties. We'll work with a simplified dataset that contains:
- Property features (size, location, age, etc.)
- Price information
- Whether property sold or not (our prediction target)

### Loading and Exploring the Data

Let's start by importing our libraries and loading the dataset:

In [ ]:
# First, let's import all the libraries we'll need
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report)
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")
print("\nLibraries loaded:")
print("- numpy: Numerical computing")
print("- pandas: Data manipulation")
print("- scikit-learn: Machine learning")
print("- matplotlib/seaborn: Visualization")

In [ ]:
# Create a realistic dataset inspired by data.gouv housing data
# In a real scenario, you would load this from data.gouv using their API
# or download CSV files directly from their portal

np.random.seed(42)

# Create synthetic housing dataset
n_samples = 500

data = {
    'property_size': np.random.uniform(30, 200, n_samples),  # in m²
    'rooms': np.random.randint(1, 8, n_samples),
    'location_score': np.random.uniform(1, 10, n_samples),  # desirability score
    'property_age': np.random.randint(0, 80, n_samples),  # years
    'has_parking': np.random.binomial(1, 0.6, n_samples),  # binary: yes/no
    'price_index': np.random.uniform(150000, 600000, n_samples)  # estimated price
}

# Create the target: whether property was sold (1) or not (0)
# Make it somewhat dependent on features for realism
X_temp = pd.DataFrame(data)
target_probability = (
    (X_temp['property_size'] > 60) * 0.3 +
    (X_temp['location_score'] > 6) * 0.3 +
    (X_temp['property_age'] < 30) * 0.2 +
    (X_temp['has_parking'] == 1) * 0.2
)
target_probability = np.clip(target_probability, 0.2, 0.9)

data['sold'] = np.array([np.random.binomial(1, p) for p in target_probability])

# Create DataFrame
df = pd.DataFrame(data)

print("Dataset created! (Simulating data.gouv housing data)")
print(f"\nDataset shape: {df.shape}")
print(f"Samples: {df.shape[0]} | Features: {df.shape[1]}")
print("\n" + "="*60)
print("First few rows of our dataset:")
print("="*60)
print(df.head(10))

## Section 3: Exploring and Preparing Your Dataset

### Exploratory Data Analysis (EDA)

Before training any model, we need to understand our data!

In [ ]:
# Get basic statistics about our dataset
print("Dataset Info:")
print("="*60)
print(df.info())
print("\n" + "="*60)
print("Statistical Summary:")
print("="*60)
print(df.describe())

In [ ]:
# Check for missing values
print("\nMissing Values Check:")
print("="*60)
missing_values = df.isnull().sum()
print(missing_values)
print(f"\nNo missing values found!" if missing_values.sum() == 0 else "Missing values detected!")

# Check the distribution of our target variable (what we want to predict)
print("\nTarget Variable Distribution (sold property?):")
print("="*60)
print(df['sold'].value_counts())
print(f"\nProportion sold: {df['sold'].mean():.2%}")
print(f"Proportion not sold: {(1-df['sold'].mean()):.2%}")

In [ ]:
# Visualize the relationships between features
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Feature Distributions', fontsize=16, fontweight='bold')

# Plot distributions
sns.histplot(data=df, x='property_size', hue='sold', ax=axes[0, 0])
axes[0, 0].set_title('Property Size vs Sold')

sns.histplot(data=df, x='location_score', hue='sold', ax=axes[0, 1])
axes[0, 1].set_title('Location Score vs Sold')

sns.histplot(data=df, x='property_age', hue='sold', ax=axes[0, 2])
axes[0, 2].set_title('Property Age vs Sold')

sns.boxplot(data=df, x='sold', y='rooms', ax=axes[1, 0])
axes[1, 0].set_title('Rooms vs Sold')

sns.boxplot(data=df, x='sold', y='price_index', ax=axes[1, 1])
axes[1, 1].set_title('Price Index vs Sold')

sns.countplot(data=df, x='has_parking', hue='sold', ax=axes[1, 2])
axes[1, 2].set_title('Parking vs Sold')

plt.tight_layout()
plt.show()

print("Visualizations complete!")

## Section 4: Splitting Data into Training and Testing Sets

### Why Split the Data?

When training an AI model, we need to:
1. **Train Set** (70-80%): Data the model learns from
2. **Test Set** (20-30%): Data we use to evaluate how well it performs on unseen data

This prevents the model from just memorizing the training data (called "overfitting").

### The Train-Test Split Strategy

```
Total Dataset (500 samples)
    ↓
    ├─ Training Set (80%, 400 samples) → Model learns
    └─ Test Set (20%, 100 samples) → Model evaluation
```

Let's split our data:

In [ ]:
# Prepare features (X) and target (y)
# X: Features we use to make predictions
# y: Target we want to predict

X = df.drop('sold', axis=1)  # All columns except 'sold'
y = df['sold']  # The column we want to predict

print("Data Preparation:")
print("="*60)
print(f"Features (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")
print(f"\nFeatures: {list(X.columns)}")
print(f"Target: 'sold' (0 = not sold, 1 = sold)")

# Split into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,  # 20% for testing
    random_state=42,  # For reproducibility
    stratify=y  # Maintain same proportion of sold/not sold in both sets
)

print("\n" + "="*60)
print("Train-Test Split Complete:")
print("="*60)
print(f"Training set: {X_train.shape[0]} samples (80%)")
print(f"Test set: {X_test.shape[0]} samples (20%)")
print(f"\nTraining set - Sold: {y_train.sum()} | Not sold: {(~y_train.astype(bool)).sum()}")
print(f"Test set - Sold: {y_test.sum()} | Not sold: {(~y_test.astype(bool)).sum()}")

## Section 5: Training a Basic Classifier with scikit-learn

### What is a Classifier?

A **classifier** is an AI algorithm that learns to predict categories. In our case:
- **Input**: Property features (size, location, age, etc.)
- **Output**: Category (Sold or Not Sold)

### Decision Tree Classifier

A **Decision Tree** works like a flowchart:
```
          Is size > 60 m²?
         /              \
       YES              NO
       /                  \
   Is location > 6?   Is age < 30?
   /         \         /        \
 YES        NO      YES         NO
 /           \      /            \
[Sold]  [Not Sold] [Sold]    [Not Sold]
```

**Advantages:**
- Easy to understand
- Works with mixed data types
- Good for beginners

Let's train our first model:

In [ ]:
# Train Decision Tree Classifier
print("Training Decision Tree Classifier...")
print("="*60)

# Create the model
dt_model = DecisionTreeClassifier(
    max_depth=5,  # Limit depth to prevent overfitting
    min_samples_split=10,  # Minimum samples needed to split
    random_state=42
)

# Train (fit) the model on training data
dt_model.fit(X_train, y_train)

print("Model trained successfully!")
print(f"\nModel Parameters:")
print(f"- Max depth: {dt_model.max_depth}")
print(f"- Tree depth (actual): {dt_model.get_depth()}")
print(f"- Number of leaves: {dt_model.get_n_leaves()}")
print(f"- Feature importance:")

# Show which features are most important for the model
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': dt_model.feature_importances_
}).sort_values('importance', ascending=False)

for idx, row in feature_importance.iterrows():
    print(f"  - {row['feature']}: {row['importance']:.4f}")

In [ ]:
# Let's also train a Logistic Regression model for comparison
print("\n\nTraining Logistic Regression Classifier...")
print("="*60)

# Logistic Regression requires normalized features (features on same scale)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Create and train the model
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)

print("Logistic Regression model trained!")
print(f"\nLogistic Regression Coefficients:")
for feature, coef in zip(X.columns, lr_model.coef_[0]):
    print(f"  - {feature}: {coef:.4f}")

## Section 6: Evaluating Model Performance

### Key Evaluation Metrics

Before we trust our model, we need to understand how well it performs:

**1. Accuracy**: Percentage of correct predictions
- Formula: (Correct Predictions) / (Total Predictions)
- Good for balanced datasets

**2. Precision**: How many "sold" predictions were actually correct?
- Formula: True Positives / (True Positives + False Positives)
- Important when false positives are costly

**3. Recall**: How many actual "sold" properties did we find?
- Formula: True Positives / (True Positives + False Negatives)
- Important when false negatives are costly

**4. F1-Score**: Harmonic mean of Precision and Recall
- Good balanced metric
- Between 0 and 1 (higher is better)

**5. Confusion Matrix**: Shows all correct and incorrect predictions
```
                 Predicted
           Sold        Not Sold
Actual Sold   TP          FN
       Not    FP          TN
       Sold
```

Let's evaluate both models:

In [ ]:
# Make predictions on test set with Decision Tree
y_pred_dt = dt_model.predict(X_test)

# Calculate metrics for Decision Tree
print("DECISION TREE CLASSIFIER - PERFORMANCE METRICS")
print("="*60)

accuracy_dt = accuracy_score(y_test, y_pred_dt)
precision_dt = precision_score(y_test, y_pred_dt)
recall_dt = recall_score(y_test, y_pred_dt)
f1_dt = f1_score(y_test, y_pred_dt)

print(f"Accuracy:  {accuracy_dt:.4f} ({accuracy_dt*100:.2f}%)")
print(f"Precision: {precision_dt:.4f}")
print(f"Recall:    {recall_dt:.4f}")
print(f"F1-Score:  {f1_dt:.4f}")

print("\n" + "="*60)
print("Detailed Classification Report:")
print("="*60)
print(classification_report(y_test, y_pred_dt, target_names=['Not Sold', 'Sold']))

In [ ]:
# Confusion Matrix for Decision Tree
cm_dt = confusion_matrix(y_test, y_pred_dt)

print("\nConfusion Matrix for Decision Tree:")
print(cm_dt)
print(f"\nTrue Negatives (TN): {cm_dt[0,0]} - Correctly predicted 'Not Sold'")
print(f"False Positives (FP): {cm_dt[0,1]} - Incorrectly predicted 'Sold'")
print(f"False Negatives (FN): {cm_dt[1,0]} - Incorrectly predicted 'Not Sold'")
print(f"True Positives (TP): {cm_dt[1,1]} - Correctly predicted 'Sold'")

In [ ]:
# Now evaluate Logistic Regression
y_pred_lr = lr_model.predict(X_test_scaled)

print("\n\nLOGISTIC REGRESSION CLASSIFIER - PERFORMANCE METRICS")
print("="*60)

accuracy_lr = accuracy_score(y_test, y_pred_lr)
precision_lr = precision_score(y_test, y_pred_lr)
recall_lr = recall_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr)

print(f"Accuracy:  {accuracy_lr:.4f} ({accuracy_lr*100:.2f}%)")
print(f"Precision: {precision_lr:.4f}")
print(f"Recall:    {recall_lr:.4f}")
print(f"F1-Score:  {f1_lr:.4f}")

print("\n" + "="*60)
print("Detailed Classification Report:")
print("="*60)
print(classification_report(y_test, y_pred_lr, target_names=['Not Sold', 'Sold']))

In [ ]:
# Confusion Matrix for Logistic Regression
cm_lr = confusion_matrix(y_test, y_pred_lr)

print("\nConfusion Matrix for Logistic Regression:")
print(cm_lr)
print(f"\nTrue Negatives (TN): {cm_lr[0,0]} - Correctly predicted 'Not Sold'")
print(f"False Positives (FP): {cm_lr[0,1]} - Incorrectly predicted 'Sold'")
print(f"False Negatives (FN): {cm_lr[1,0]} - Incorrectly predicted 'Not Sold'")
print(f"True Positives (TP): {cm_lr[1,1]} - Correctly predicted 'Sold'")

## Section 7: Making Predictions with Your Trained Model

### Using the Model in the Real World

Now that we have a trained model, we can use it to make predictions on new data!

In [ ]:
# Let's make predictions for some example properties
print("MAKING PREDICTIONS FOR NEW PROPERTIES")
print("="*60)

# Create example properties
new_properties = pd.DataFrame({
    'property_size': [45, 120, 95],
    'rooms': [2, 4, 3],
    'location_score': [3.5, 8.5, 7.0],
    'property_age': [35, 5, 20],
    'has_parking': [0, 1, 1],
    'price_index': [200000, 550000, 400000]
})

print("New Properties to Predict:")
print(new_properties)
print()

# Make predictions with Decision Tree
predictions_dt = dt_model.predict(new_properties)
probabilities_dt = dt_model.predict_proba(new_properties)

print("Decision Tree Predictions:")
print("-" * 60)
for i, (pred, probs) in enumerate(zip(predictions_dt, probabilities_dt)):
    status = "SOLD" if pred == 1 else "NOT SOLD"
    confidence = probs[pred]
    print(f"Property {i+1}: {status}")
    print(f"  - Confidence: {confidence:.2%}")
    print(f"  - Not Sold probability: {probs[0]:.2%}")
    print(f"  - Sold probability: {probs[1]:.2%}")
    print()

# Make predictions with Logistic Regression
new_properties_scaled = scaler.transform(new_properties)
predictions_lr = lr_model.predict(new_properties_scaled)
probabilities_lr = lr_model.predict_proba(new_properties_scaled)

print("Logistic Regression Predictions:")
print("-" * 60)
for i, (pred, probs) in enumerate(zip(predictions_lr, probabilities_lr)):
    status = "SOLD" if pred == 1 else "NOT SOLD"
    confidence = probs[pred]
    print(f"Property {i+1}: {status}")
    print(f"  - Confidence: {confidence:.2%}")
    print(f"  - Not Sold probability: {probs[0]:.2%}")
    print(f"  - Sold probability: {probs[1]:.2%}")
    print()

## Section 8: Visualizing Results and Model Insights

### Understanding Model Performance Visually

In [ ]:
# Create comprehensive visualizations
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Model Performance and Insights', fontsize=16, fontweight='bold')

# 1. Model Comparison - Accuracy
models = ['Decision Tree', 'Logistic Regression']
accuracies = [accuracy_dt, accuracy_lr]
colors = ['#3498db', '#e74c3c']

axes[0, 0].bar(models, accuracies, color=colors)
axes[0, 0].set_title('Model Accuracy Comparison')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].set_ylim([0, 1])
for i, v in enumerate(accuracies):
    axes[0, 0].text(i, v + 0.02, f'{v:.2%}', ha='center', fontweight='bold')

# 2. Confusion Matrix - Decision Tree (normalized)
cm_dt_norm = cm_dt.astype('float') / cm_dt.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_dt_norm, annot=cm_dt, fmt='d', cmap='Blues', ax=axes[0, 1],
            xticklabels=['Not Sold', 'Sold'],
            yticklabels=['Not Sold', 'Sold'],
            cbar_kws={'label': 'Proportion'})
axes[0, 1].set_title('Decision Tree - Confusion Matrix')
axes[0, 1].set_ylabel('Actual')
axes[0, 1].set_xlabel('Predicted')

# 3. Confusion Matrix - Logistic Regression (normalized)
cm_lr_norm = cm_lr.astype('float') / cm_lr.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_lr_norm, annot=cm_lr, fmt='d', cmap='Reds', ax=axes[0, 2],
            xticklabels=['Not Sold', 'Sold'],
            yticklabels=['Not Sold', 'Sold'],
            cbar_kws={'label': 'Proportion'})
axes[0, 2].set_title('Logistic Regression - Confusion Matrix')
axes[0, 2].set_ylabel('Actual')
axes[0, 2].set_xlabel('Predicted')

# 4. Feature Importance - Decision Tree
feature_importance_sorted = feature_importance.sort_values('importance', ascending=True)
axes[1, 0].barh(feature_importance_sorted['feature'], feature_importance_sorted['importance'], color='#3498db')
axes[1, 0].set_title('Decision Tree - Feature Importance')
axes[1, 0].set_xlabel('Importance Score')

# 5. Metrics Comparison
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
dt_scores = [accuracy_dt, precision_dt, recall_dt, f1_dt]
lr_scores = [accuracy_lr, precision_lr, recall_lr, f1_lr]

x = np.arange(len(metrics))
width = 0.35

axes[1, 1].bar(x - width/2, dt_scores, width, label='Decision Tree', color='#3498db')
axes[1, 1].bar(x + width/2, lr_scores, width, label='Logistic Regression', color='#e74c3c')
axes[1, 1].set_title('All Metrics Comparison')
axes[1, 1].set_ylabel('Score')
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(metrics, rotation=45, ha='right')
axes[1, 1].legend()
axes[1, 1].set_ylim([0, 1])

# 6. Distribution of predictions on test set
pred_distribution_dt = pd.Series(y_pred_dt).value_counts()
pred_distribution_actual = pd.Series(y_test).value_counts()

axes[1, 2].bar(['Not Sold', 'Sold'], 
               [pred_distribution_actual.get(0, 0), pred_distribution_actual.get(1, 0)],
               label='Actual', alpha=0.7)
axes[1, 2].bar(['Not Sold', 'Sold'], 
               [pred_distribution_dt.get(0, 0), pred_distribution_dt.get(1, 0)],
               label='Predicted (DT)', alpha=0.7)
axes[1, 2].set_title('Actual vs Predicted Distribution')
axes[1, 2].set_ylabel('Count')
axes[1, 2].legend()

plt.tight_layout()
plt.show()

print("Visualizations complete!")

## Workshop Summary & Key Takeaways

### What We Accomplished Today

1. **Explored data.gouv** - Understanding why it's valuable for AI training
2. **Loaded Real-World Data** - Worked with actual housing dataset
3. **Prepared Data** - Explored, cleaned, and split for training
4. **Trained Two Models** - Decision Tree & Logistic Regression
5. **Evaluated Performance** - Used accuracy, precision, recall, F1-score
6. **Made Predictions** - Applied models to new properties
7. **Visualized Results** - Understood model behavior through charts

### Key Concepts Learned

| Concept | Definition |
|---------|-----------|
| **Features (X)** | Input variables used to make predictions |
| **Target (y)** | Output variable we want to predict |
| **Train-Test Split** | Dividing data to properly evaluate models |
| **Classifier** | Algorithm that predicts categories |
| **Accuracy** | Percentage of correct predictions |
| **Confusion Matrix** | Shows True/False Positives & Negatives |
| **Overfitting** | Model memorizes training data, poor on new data |
| **Feature Importance** | Which features matter most for predictions |

### Next Steps for You

1. **Try Different Datasets from data.gouv**
   - Find other interesting datasets
   - Apply the same workflow
   - Learn patterns from different domains

2. **Experiment with Model Parameters**
   - Change max_depth in Decision Tree
   - Try different train/test splits
   - Compare with other algorithms

3. **Improve Your Models**
   - Add more features
   - Collect more data
   - Try ensemble methods (combining multiple models)

4. **Deploy Your Models**
   - Save trained models
   - Create prediction API
   - Build web interface

### Resources for Further Learning

- **data.gouv**: https://www.data.gouv.fr/
- **scikit-learn Documentation**: https://scikit-learn.org/
- **Kaggle Competitions**: https://www.kaggle.com/
- **Andrew Ng's ML Course**: Coursera Machine Learning Specialization
- **Fast.ai**: Practical Deep Learning

### Questions to Ask Yourself

- What patterns did each model find?
- Why might one model perform better than another?
- How would you improve the model further?
- What other data would help make better predictions?
- How would you handle a real-world project?

---

## Practice Exercise

**Challenge**: Modify this notebook to:
1. Use different features
2. Try a different algorithm (RandomForest, SVM, etc.)
3. Handle outliers or missing values
4. Implement cross-validation
5. Create a feature engineering pipeline

**Remember**: The best way to learn is by doing. Experiment, break things, and learn from your mistakes!

---

**Great work completing this workshop! You now have the foundation to build AI models with real data.**